# DDDQN Training For Dynamic Malware Analysis

In [7]:
# Environment Setup: Import required dependencies for reinforcement learning pipeline
import os
import json
import random
import numpy as np
import pandas as pd
from collections import deque, defaultdict
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from enum import IntEnum
import pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Deep Learning Framework: PyTorch for neural network implementation
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Reproducibility Configuration: Set deterministic seeds across all libraries
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

# System Diagnostics: Display computational environment details
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu130
CUDA available: False


In [6]:
# Action Space Definition: Discrete actions for sequential malware analysis
class Action(IntEnum):
    """
    Enumeration of available actions in the reinforcement learning environment.
    
    Actions 0-4 represent investigative operations with incremental information revelation.
    Actions 5-6 are terminal classification decisions.
    """
    CONTINUE = 0          # Baseline analysis: Always-available metadata and basic features
    FOCUS_MEMORY = 1      # Memory analysis: Process events and injection indicators
    FOCUS_FILESYSTEM = 2  # Filesystem analysis: File operations and dropped artifacts
    FOCUS_NETWORK = 3     # Network analysis: DNS queries, HTTP requests, TCP/UDP connections
    MEMORY_DUMP = 4       # Deep memory inspection: Anomalies, encrypted buffers, payloads
    TERMINATE_MALWARE = 5 # Terminal action: Classify sample as MALWARE
    TERMINATE_BENIGN = 6  # Terminal action: Classify sample as BENIGN

NUM_ACTIONS = len(Action)
TERMINAL_ACTIONS = {Action.TERMINATE_MALWARE, Action.TERMINATE_BENIGN}

# Cost Structure: Computational penalties for different analysis operations
# Reflects real-world resource consumption in sandbox environments
ACTION_COSTS = {
    Action.CONTINUE: 0.1,
    Action.FOCUS_MEMORY: 0.5,
    Action.FOCUS_FILESYSTEM: 0.5,
    Action.FOCUS_NETWORK: 0.5,
    Action.MEMORY_DUMP: 1.0,  # Highest cost: Deep memory analysis
    Action.TERMINATE_MALWARE: 0.0,
    Action.TERMINATE_BENIGN: 0.0
}

## Dataset Loading With Memory-Efficient Technique

Lazy-loading mechanism for MalWhere dataset to manage memory constraints when processing large size reports.

### Dataset Characteristics:
- **Source**: MalWhere Dataset
- **Format**: MalWhere JSON reports
- **Classes**: Binary classification (BENIGN vs MALWARE)

### Implementation Strategy:
Rather than loading all reports into memory the `LazyReportLoader` class maintains file paths and loads reports on-demand with LRU caching for frequently accessed samples. This approach enables training on large datasets without OOM problems

In [ ]:
# Dataset Loader: Memory-efficient lazy loading for CAPE sandbox reports
def load_cape_dataset_lazy():
    """
    Initialize dataset paths without loading full reports into memory.
    
    Returns:
        report_paths (list): File system paths to individual JSON reports
        labels (list): Corresponding ground truth labels (BENIGN/MALWARE)
        label_mapping (dict): Optional mapping file for additional metadata
    """
    BASE_PATH = "/kaggle/input/winmet-windows-malware-execution-traces-dataset"
    BENIGN_PATH = os.path.join(BASE_PATH, "BENIGN")
    MALWARE_PATH = os.path.join(BASE_PATH, "MALWARE")
    MAPPING_PATH = os.path.join(BASE_PATH, "cape_report_to_label_mapping.json")
    
    report_paths = []
    labels = []
    
    # Load auxiliary label mapping if available
    try:
        with open(MAPPING_PATH, 'r') as f:
            label_mapping = json.load(f)
        print(f"Loaded label mapping with {len(label_mapping)} entries")
    except Exception as e:
        print(f"Warning: Could not load mapping file: {e}")
        label_mapping = {}
    
    # Utility function: Collect file paths from directory structure
    def collect_folder_paths(folder_path, label_type):
        folder_paths = []
        folder_labels = []
        
        print(f"Collecting {label_type} report paths from {folder_path}...")
        
        for filename in os.listdir(folder_path):
            if filename.endswith('.json'):
                file_path = os.path.join(folder_path, filename)
                folder_paths.append(file_path)
                folder_labels.append(label_type)
        
        return folder_paths, folder_labels
    
    # Collect benign sample paths
    if os.path.exists(BENIGN_PATH):
        benign_paths, benign_labels = collect_folder_paths(BENIGN_PATH, 'BENIGN')
        report_paths.extend(benign_paths)
        labels.extend(benign_labels)
    else:
        print(f"Warning: BENIGN folder not found at {BENIGN_PATH}")
    
    # Collect malware sample paths
    if os.path.exists(MALWARE_PATH):
        malware_paths, malware_labels = collect_folder_paths(MALWARE_PATH, 'MALWARE')
        report_paths.extend(malware_paths)
        labels.extend(malware_labels)
    else:
        print(f"Warning: MALWARE folder not found at {MALWARE_PATH}")
    
    print(f"\nDataset Summary:")
    print(f"  Total reports: {len(report_paths)}")
    print(f"  Benign: {labels.count('BENIGN')}")
    print(f"  Malware: {labels.count('MALWARE')}")
    
    return report_paths, labels, label_mapping

# Lazy Report Loader: On-demand loading with caching
class LazyReportLoader:
    """
    Memory-efficient report loader with LRU caching.
    
    Attributes:
        report_paths (list): Paths to JSON report files
        labels (list): Ground truth labels
        cache (dict): LRU cache for frequently accessed reports
        cache_size (int): Maximum number of reports to cache in memory
    """
    def __init__(self, report_paths, labels, cache_size=50):
        self.report_paths = report_paths
        self.labels = labels
        self.cache = {}  # LRU cache implementation
        self.cache_size = cache_size
        
    def get_report(self, idx):
        """
        Retrieve report by index with caching mechanism.
        
        Args:
            idx (int): Index of report in dataset
            
        Returns:
            dict: Parsed JSON report
        """
        if idx in self.cache:
            return self.cache[idx]
        
        # Load from disk
        with open(self.report_paths[idx], 'r') as f:
            report = json.load(f)
        
        # Update cache with FIFO eviction policy
        if len(self.cache) >= self.cache_size:
            oldest_key = next(iter(self.cache))
            del self.cache[oldest_key]
        
        self.cache[idx] = report
        return report
    
    def __len__(self):
        return len(self.report_paths)
    
    def __getitem__(self, idx):
        return self.get_report(idx), self.labels[idx]

# Initialize dataset infrastructure
report_paths, labels, label_mapping = load_cape_dataset_lazy()
report_loader = LazyReportLoader(report_paths, labels, cache_size=100)

print(f"\nLazy loader created with {len(report_loader)} reports")
print(f"Cache size: {report_loader.cache_size} reports")

## Feature Extraction Pipeline

This pipline is an efficient approach to use feature engineering instead of using all available informations in MalWhere reports. We create a structured numerical features to train our model.

### Feature Engineering Approach:
The extractor implements a hierarchical feature extraction strategy aligned with the action space:
- **Basic Metadata**: Metadata (file size, type, process counts)
- **Memory Features**: Process behavior and injection indicators
- **Filesystem Features**: File operations and dropped artifacts
- **Network Features**: Network activity patterns and API calls
- **Memory Dump Features**: Deep behavioral analysis (anomalies, signatures, payloads)

In [ ]:
class CAPEFeatureExtractor:
    from collections import defaultdict
    """
    Comprehensive feature extraction system for CAPE v2 malware analysis reports.
    
    This class implements hierarchical feature extraction corresponding to the 
    reinforcement learning action space, enabling progressive information revelation.
    
    Design Pattern: Static methods allow stateless operation for parallel processing.
    """
    
    # Domain Knowledge: API calls indicative of code injection techniques
    INJECTION_APIS = {
        'WriteProcessMemory', 'VirtualAllocEx', 'CreateRemoteThread',
        'NtWriteVirtualMemory', 'RtlCreateUserThread', 'QueueUserAPC'
    }
    
    @staticmethod
    def safe_get(data: Dict, keys: List[str], default=0):
        """
        Safely navigate nested dictionary structures with fallback defaults.
        
        Args:
            data: Source dictionary
            keys: List of keys representing nested path
            default: Default value if path doesn't exist
            
        Returns:
            Retrieved value or default
        """
        current = data
        for key in keys:
            if isinstance(current, dict) and key in current:
                current = current[key]
            else:
                return default
        return current
    
    @classmethod
    def extract_basic_features(cls, report: Dict) -> Dict[str, float]:
        """
        Extract baseline features available without expensive operations (Action 0).
        
        Feature Categories:
        - File metadata: Size, type, CAPE classification code
        - Process information: Count, threads, tree complexity
        - API call statistics: Total invocations
        
        Args:
            report: CAPE v2 JSON report
            
        Returns:
            Dictionary of normalized feature values
        """
        features = {}
        
        # Extract file metadata from target section
        target = report.get('target', {})
        file_info = target.get('file', {})
        
        features['file_size'] = float(file_info.get('size', 0))
        features['cape_type_code'] = float(file_info.get('cape_type_code', 0))
        
        # Encode file type as ordinal variable
        file_type = str(file_info.get('type', '')).lower()
        if 'pe32' in file_type or 'exe' in file_type:
            features['file_type_ord'] = 1.0
        elif 'dll' in file_type:
            features['file_type_ord'] = 2.0
        else:
            features['file_type_ord'] = 0.0
        
        # Behavioral indicators from process monitoring
        behavior = report.get('behavior', {})
        processes = behavior.get('processes', [])
        
        features['n_processes'] = float(len(processes))
        
        # Aggregate thread count across all processes
        thread_count = 0
        for proc in processes:
            threads = proc.get('threads', [])
            thread_count += len(threads)
        features['n_threads_total'] = float(thread_count)
        
        # Process tree complexity metric
        processtree = behavior.get('processtree', [])
        features['proc_tree_nodes'] = float(cls._count_tree_nodes(processtree))
        
        # API invocation frequency
        total_api_calls = 0
        for proc in processes:
            calls = proc.get('calls', [])
            total_api_calls += len(calls)
        features['total_api_calls'] = float(total_api_calls)
        
        return features
    
    @classmethod
    def extract_memory_features(cls, report: Dict) -> Dict[str, float]:
        """
        Extract memory-related behavioral indicators (Action 1).
        
        Feature Categories:
        - Enhanced memory events from CAPE monitoring
        - Code injection pattern detection via API analysis
        
        Args:
            report: CAPE v2 JSON report
            
        Returns:
            Dictionary of memory-specific features
        """
        features = {}
        behavior = report.get('behavior', {})
        processes = behavior.get('processes', [])
        
        # Enhanced monitoring events
        enhanced = behavior.get('enhanced', [])
        features['n_enhanced_events'] = float(len(enhanced))
        
        # Injection detection: Count suspicious API invocations
        injection_count = 0
        for proc in processes:
            calls = proc.get('calls', [])
            for call in calls:
                api = call.get('api', '')
                if any(inj_api in api for inj_api in cls.INJECTION_APIS):
                    injection_count += 1
        
        features['n_injection_indicators'] = float(injection_count)
        
        return features
    
    @classmethod
    def extract_filesystem_features(cls, report: Dict) -> Dict[str, float]:
        """
        Extract filesystem interaction patterns (Action 2).
        
        Feature Categories:
        - File operations: Read, write, delete counts
        - Dropped artifacts: Count and cumulative size
        
        Args:
            report: CAPE v2 JSON report
            
        Returns:
            Dictionary of filesystem features
        """
        features = {}
        behavior = report.get('behavior', {})
        summary = behavior.get('summary', {})
        
        # File operation statistics from behavior summary
        features['files_total'] = float(len(summary.get('files', [])))
        features['read_files'] = float(len(summary.get('read_files', [])))
        features['write_files'] = float(len(summary.get('write_files', [])))
        features['delete_files'] = float(len(summary.get('delete_files', [])))
        
        # Dropped file analysis
        dropped = report.get('dropped', [])
        features['n_dropped'] = float(len(dropped))
        dropped_sizes = [d.get('size', 0) for d in dropped]
        features['dropped_total_size'] = float(sum(dropped_sizes))
        
        return features
    
    @classmethod
    def extract_network_features(cls, report: Dict) -> Dict[str, float]:
        """
        Extract network activity patterns from API call analysis (Action 3).
        
        Feature Engineering Strategy:
        Instead of relying on CAPE's network section (which may be incomplete),
        this method analyzes API calls to identify network-related behavior.
        
        Feature Categories:
        - API call classification: DNS, HTTP, socket operations
        - Network activity ratios: Relative frequency of network operation types
        - Signature-based indicators: Network-related CAPE signatures
        
        Args:
            report: CAPE v2 JSON report
            
        Returns:
            Dictionary of network behavior features
        """
        features = {}
        behavior = report.get('behavior', {})
        processes = behavior.get('processes', [])
        
        # Initialize counters for different network operation types
        total_network_calls = 0
        network_calls_by_type = defaultdict(int)
        
        # API categorization based on Windows networking functions
        dns_apis = ['dns', 'gethost', 'getaddrinfo', 'getnameinfo']
        http_apis = ['http', 'internet', 'winhttp', 'url']
        socket_apis = ['socket', 'connect', 'bind', 'listen', 'accept', 
                        'send', 'recv', 'closesocket', 'wsa']
        crypto_url_apis = ['cryptretrieveobjectbyurl', 'urlcanonicalize']
        network_apis = ['ras', 'getadaptersaddresses', 'setsockopt', 
                        'ioctlsocket', 'wsa', 'getsockopt']
        
        # Iterate through all API calls and classify network operations
        for proc in processes:
            calls = proc.get('calls', [])
            for call in calls:
                api = call.get('api', '').lower()
                category = call.get('category', '').lower()
                
                is_network_call = False
                
                # Classification by CAPE category
                if category == 'network':
                    is_network_call = True
                
                # Classification by API function name pattern matching
                elif any(net_api in api for net_api in dns_apis):
                    network_calls_by_type['dns'] += 1
                    is_network_call = True
                elif any(net_api in api for net_api in http_apis):
                    network_calls_by_type['http'] += 1
                    is_network_call = True
                elif any(net_api in api for net_api in socket_apis):
                    network_calls_by_type['socket'] += 1
                    is_network_call = True
                elif any(net_api in api for net_api in crypto_url_apis):
                    network_calls_by_type['crypto_url'] += 1
                    is_network_call = True
                elif any(net_api in api for net_api in network_apis):
                    network_calls_by_type['other_network'] += 1
                    is_network_call = True
                
                if is_network_call:
                    total_network_calls += 1
        
        # Absolute frequency features
        features['total_network_calls'] = float(total_network_calls)
        features['dns_calls'] = float(network_calls_by_type.get('dns', 0))
        features['http_calls'] = float(network_calls_by_type.get('http', 0))
        features['socket_calls'] = float(network_calls_by_type.get('socket', 0))
        features['crypto_url_calls'] = float(network_calls_by_type.get('crypto_url', 0))
        features['other_network_calls'] = float(network_calls_by_type.get('other_network', 0))
        
        # Relative frequency features (normalized by total network activity)
        if total_network_calls > 0:
            features['dns_ratio'] = features['dns_calls'] / total_network_calls
            features['http_ratio'] = features['http_calls'] / total_network_calls
            features['socket_ratio'] = features['socket_calls'] / total_network_calls
            features['crypto_url_ratio'] = features['crypto_url_calls'] / total_network_calls
        else:
            features['dns_ratio'] = 0.0
            features['http_ratio'] = 0.0
            features['socket_ratio'] = 0.0
            features['crypto_url_ratio'] = 0.0
        
        # Signature-based network behavior indicators
        network_signatures = 0
        signatures = report.get('signatures', [])
        for sig in signatures:
            name = sig.get('name', '').lower()
            if any(net_term in name for net_term in 
                    ['network', 'dns', 'http', 'c2', 'botnet', 'communication', 
                    'download', 'upload', 'socket', 'connection']):
                network_signatures += 1
        
        features['network_signatures'] = float(network_signatures)
        
        return features
    
    @classmethod
    def extract_memory_dump_features(cls, report: Dict) -> Dict[str, float]:
        """
        Extract deep behavioral analysis features requiring memory dumps (Action 4).
        
        This represents the most computationally expensive analysis level,
        providing detailed insights into malware behavior.
        
        Feature Categories:
        - Behavioral anomalies detected by CAPE
        - Encrypted buffer analysis
        - CAPE signature matches and alert levels
        - Extracted payload characteristics
        
        Args:
            report: CAPE v2 JSON report
            
        Returns:
            Dictionary of deep analysis features
        """
        features = {}
        behavior = report.get('behavior', {})
        
        # Behavioral anomaly detection
        anomalies = behavior.get('anomaly', [])
        features['n_anomalies'] = float(len(anomalies))
        
        # Encrypted buffer detection (potential crypter/packer indicators)
        encrypted = behavior.get('encryptedbuffers', [])
        features['n_encryptedbuffers'] = float(len(encrypted))
        
        # CAPE signature analysis
        signatures = report.get('signatures', [])
        alert_signatures = [s for s in signatures if s.get('alert', False)]
        features['n_signatures'] = float(len(signatures))
        features['signatures_alert_count'] = float(len(alert_signatures))
        
        # Payload extraction analysis
        cape_section = report.get('CAPE', {})
        payloads = cape_section.get('payloads', [])
        features['n_payloads'] = float(len(payloads))
        payload_sizes = [p.get('size', 0) for p in payloads]
        features['payloads_total_size'] = float(sum(payload_sizes))
        
        return features
    
    @staticmethod
    def _count_tree_nodes(tree: List) -> int:
        """
        Recursively compute process tree complexity metric.
        
        Args:
            tree: Process tree structure from CAPE report
            
        Returns:
            Total node count in process tree
        """
        count = 0
        for node in tree:
            count += 1
            count += CAPEFeatureExtractor._count_tree_nodes(node.get('children', []))
        return count

# Validation: Test feature extraction on sample report
if 'report_loader' in locals() and len(report_loader) > 0:
    print("Validating feature extraction pipeline on sample report...")
    test_report = report_loader.get_report(0)
    
    # Execute all feature extractors
    test_basic = CAPEFeatureExtractor.extract_basic_features(test_report)
    test_network = CAPEFeatureExtractor.extract_network_features(test_report)
    
    print(f"Basic features extracted: {len(test_basic)} features")
    print(f"Network features extracted: {len(test_network)} features")
    
    print("\nNetwork Feature Sample:")
    for key, value in test_network.items():
        print(f"  {key}: {value}")

## State Space Construction
MalWhere Agent is state is simply the features vector with added metadata features. Note that we adopt slots in our state to make the agent decides which next slot should be revealed to mimic the human behavior in dynamic analysis.

### State Components 
- Action slots: slots for each action features
- Revealed info mask: binary indicators showing which analyses have been performed
- Temporal context: step count and last action taken
- Normalization: feature scaling [0, 1] for stable and fast learning process

In [ ]:
# State Builder: Construct fixed-dimensional state representations with action-specific slots
class StateBuilder:
    """
    State representation builder for reinforcement learning environment.
    
    Architecture: Slot-based design where each action type has pre-allocated dimensions.
    This allows the neural network to learn which actions revealed which information.
    
    State Vector Structure:
    [Action_0_features | Action_1_features | ... | Action_4_features | metadata]
    
    Unrevealed action slots are zero-padded, creating a sparse representation that
    the network learns to interpret as missing information.
    """
    
    # Feature dimensionality for each action (empirically determined from extractor output)
    FEATURE_DIMS = {
        Action.CONTINUE: 6,           # Basic metadata and process counts
        Action.FOCUS_MEMORY: 3,       # Memory events and injection indicators
        Action.FOCUS_FILESYSTEM: 6,   # File operations and dropped artifacts
        Action.FOCUS_NETWORK: 11,     # Network API analysis and signatures
        Action.MEMORY_DUMP: 7,        # Deep behavioral analysis
    }
    
    TOTAL_FEATURE_DIM = sum(FEATURE_DIMS.values()) + 2  # +2 for temporal metadata (step_id, last_action)
    
    @classmethod
    def build_state(cls, 
                    report: Dict, 
                    revealed_actions: set, 
                    step_id: int, 
                    last_action: Optional[int] = None) -> np.ndarray:
        """
        Construct state vector from CAPE report based on revealed information.
        
        The state vector encodes both the extracted features and which analyses
        have been performed, enabling the agent to reason about information gaps.
        
        Args:
            report: CAPE v2 JSON report
            revealed_actions: Set of actions already taken (features available)
            step_id: Current timestep in episode
            last_action: Previous action taken (None for initial state)
            
        Returns:
            numpy.ndarray: Normalized state vector of dimension TOTAL_FEATURE_DIM
        """
        extractor = CAPEFeatureExtractor
        state_parts = []
        
        # Slot 0: Basic features (always revealed at episode start)
        feat0 = extractor.extract_basic_features(report)
        state_parts.extend(cls._normalize_features(feat0, Action.CONTINUE))
        
        # Slot 1: Memory features (revealed if Action 1 taken)
        if Action.FOCUS_MEMORY in revealed_actions:
            feat1 = extractor.extract_memory_features(report)
            state_parts.extend(cls._normalize_features(feat1, Action.FOCUS_MEMORY))
        else:
            state_parts.extend([0.0] * cls.FEATURE_DIMS[Action.FOCUS_MEMORY])
        
        # Slot 2: Filesystem features (revealed if Action 2 taken)
        if Action.FOCUS_FILESYSTEM in revealed_actions:
            feat2 = extractor.extract_filesystem_features(report)
            state_parts.extend(cls._normalize_features(feat2, Action.FOCUS_FILESYSTEM))
        else:
            state_parts.extend([0.0] * cls.FEATURE_DIMS[Action.FOCUS_FILESYSTEM])
        
        # Slot 3: Network features (revealed if Action 3 taken)
        if Action.FOCUS_NETWORK in revealed_actions:
            feat3 = extractor.extract_network_features(report)
            state_parts.extend(cls._normalize_features(feat3, Action.FOCUS_NETWORK))
        else:
            state_parts.extend([0.0] * cls.FEATURE_DIMS[Action.FOCUS_NETWORK])
        
        # Slot 4: Memory dump features (revealed if Action 4 taken)
        if Action.MEMORY_DUMP in revealed_actions:
            feat4 = extractor.extract_memory_dump_features(report)
            state_parts.extend(cls._normalize_features(feat4, Action.MEMORY_DUMP))
        else:
            state_parts.extend([0.0] * cls.FEATURE_DIMS[Action.MEMORY_DUMP])
        
        # Temporal metadata (normalized to [0,1] range)
        state_parts.append(float(step_id) / 20.0)  # Normalized step count (max 20 steps assumed)
        state_parts.append(float(last_action if last_action is not None else -1) / 10.0)
        
        return np.array(state_parts, dtype=np.float32)
    
    @staticmethod
    def _normalize_features(features: Dict[str, float], action: Action) -> List[float]:
        """
        Apply feature-specific normalization strategies.
        
        Different feature types require different normalization approaches:
        - Size/count features: Logarithmic scaling to handle wide dynamic range
        - Binary indicators: Direct scaling
        - Categorical features: Ordinal encoding
        
        Args:
            features: Raw feature dictionary
            action: Action type (determines expected dimension)
            
        Returns:
            List of normalized feature values
        """
        normalized = []
        
        # Feature-specific normalization strategies
        for key, value in features.items():
            if 'size' in key or 'total' in key:
                # Logarithmic normalization for size-based features (handles exponential distributions)
                if value > 0:
                    normalized.append(np.log1p(value) / 15.0)  # log(1+x)/15 scales large values
                else:
                    normalized.append(0.0)
            elif 'count' in key or 'n_' in key:
                # Linear normalization with saturation for count features
                normalized.append(min(value / 100.0, 1.0))
            else:
                # Default normalization for categorical and other features
                normalized.append(min(value / 10.0, 1.0))
        
        # Dimension alignment: Pad or truncate to match expected slot size
        expected_dim = StateBuilder.FEATURE_DIMS[action]
        if len(normalized) < expected_dim:
            normalized.extend([0.0] * (expected_dim - len(normalized)))
        elif len(normalized) > expected_dim:
            normalized = normalized[:expected_dim]
        
        return normalized

# Validation: Test state construction
if 'report_loader' in locals() and len(report_loader) > 0:
    print("\nValidating state builder...")
    test_report = report_loader.get_report(0)
    test_state = StateBuilder.build_state(
        report=test_report,
        revealed_actions={Action.CONTINUE},
        step_id=0,
        last_action=None
    )
    print(f"State dimension: {test_state.shape}")
    print(f"Expected dimension: {StateBuilder.TOTAL_FEATURE_DIM}")
    print(f"Sample state values (first 10): {test_state[:10]}")

## MalWhere Gym 
- State space: N-dim vector
- Action space: 7 discrete actions
- Reward: rewards with step penalties
- Episode Termination: terminal actions or max step limit

### Reward Engineering:
The reward function balances multiple objectives:
1. **Classification Accuracy**: large positive reward for correct classifications
2. **Efficiency**: step penalties encourage faster decision-making
3. **Cost Awareness**: action specific costs reflect real-world resource consumption
4. **Exploration Incentives**: small rewards for revealing informative features

In [ ]:
# Episode Result Tracking: Data structure for experiment logging
@dataclass
class EpisodeResult:
    """
    Comprehensive episode outcome record for post-training analysis.
    
    Attributes:
        report_id: Unique identifier for analyzed sample
        label: Ground truth classification
        steps: Number of actions taken before termination
        total_reward: Cumulative reward over episode
        correct: Binary correctness indicator
        actions_taken: Sequence of actions executed
        final_state: Terminal state vector
    """
    report_id: str
    label: str
    steps: int
    total_reward: float
    correct: bool
    actions_taken: List[int]
    final_state: np.ndarray

In [ ]:
class CapeMalwareEnv:
    """
    OpenAI Gym-style environment for malware analysis with progressive information revelation.
    
    This environment simulates the sequential decision-making process of a malware analyst,
    where expensive operations (memory dumps, deep analysis) can be deferred until necessary.
    
    Design Principles:
    - Sparse Rewards: Large rewards only at episode termination
    - Cost-Based Shaping: Action costs reflect computational overhead
    - Progressive Revelation: Features are revealed incrementally based on actions
    - Robustness: Handles class imbalance via balanced sampling
    """
    
    def __init__(self, report_loader: LazyReportLoader, max_steps: int = 20):
        """
        Initialize environment with dataset loader and episode constraints.
        
        Args:
            report_loader: LazyReportLoader instance for accessing CAPE reports
            max_steps: Maximum actions per episode (prevents infinite loops)
        """
        self.report_loader = report_loader
        self.max_steps = max_steps
        
        # Class balancing: Compute weights for reward scaling
        self.class_weights = self._compute_class_weights([report_loader.labels[i] for i in range(len(report_loader))])
        
        # Episode state tracking
        self.current_report_idx = None
        self.current_report = None
        self.current_label = None
        self.revealed_actions = set()
        self.step_count = 0
        self.done = False
        self.total_reward = 0.0
        
        # Trajectory logging for analysis
        self.action_history = []
        self.feature_history = []
    
    def _compute_class_weights(self, labels: List[str]) -> Dict[str, float]:
        """
        Calculate class weights for balanced reward assignment.
        
        Args:
            labels: List of all sample labels
            
        Returns:
            Dictionary mapping class labels to weight factors
        """
        return {'BENIGN': 1.0, 'MALWARE': 1.0}  # Balanced weighting
    
    def reset(self, idx: Optional[int] = None) -> np.ndarray:
        """
        Reset environment to initial state for new episode.
        
        Args:
            idx: Optional specific sample index (None for random selection)
            
        Returns:
            Initial state vector
        """
        if idx is None:
            self.current_report_idx = random.randint(0, len(self.report_loader) - 1)
        else:
            self.current_report_idx = idx
            
        # Lazy load the selected report
        self.current_report, self.current_label = self.report_loader[self.current_report_idx]
        
        # Initialize episode tracking variables
        self.revealed_actions = {Action.CONTINUE}  # Basic features always available
        self.step_count = 0
        self.done = False
        self.total_reward = 0.0
        self.action_history = []
        self.feature_history = []
        
        # Construct initial state (only basic features revealed)
        state = StateBuilder.build_state(
            self.current_report, 
            self.revealed_actions, 
            self.step_count,
            last_action=None
        )
        
        return state
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        """
        Execute action and transition to next state.
        
        Implements the core MDP transition function: s' = T(s, a)
        
        Args:
            action: Integer action index
            
        Returns:
            Tuple of (next_state, reward, done, info_dict)
        """
        if self.done:
            raise ValueError("Episode terminated. Call reset() to start new episode.")
            
        action_enum = Action(action)
        reward = 0.0
        info = {
            'label': self.current_label, 
            'step': self.step_count,
            'report_idx': self.current_report_idx
        }
        
        # Log action for trajectory analysis
        self.action_history.append(action)
        
        # Apply step penalty (encourages efficiency)
        reward -= 0.1
        
        # Apply action-specific computational cost
        if action_enum in ACTION_COSTS:
            reward -= ACTION_COSTS[action_enum]
        
        # Terminal action handling: Classification decision
        if action_enum in TERMINAL_ACTIONS:
            self.done = True
            predicted_malware = (action_enum == Action.TERMINATE_MALWARE)
            actual_malware = (self.current_label == 'MALWARE')
            
            if predicted_malware == actual_malware:
                # Correct classification: Large positive reward
                class_weight = self.class_weights.get(self.current_label, 1.0)

                if not actual_malware:  # True Negative
                    reward += 15.0 * class_weight
                else:  # True Positive
                    reward += 15.0 * class_weight
                
                info['correct'] = True
                info['termination'] = 'correct'
            else:
                # Misclassification: Large negative penalty
                if not actual_malware:  # False Positive
                    reward -= 25.0
                else:  # False Negative (more critical in security context)
                    reward -= 25.0
                
                info['correct'] = False
                info['termination'] = 'incorrect'
                
            info['terminated'] = True
            info['prediction'] = 'MALWARE' if predicted_malware else 'BENIGN'
            
        else:
            # Non-terminal action: Information gathering
            if action_enum in self.revealed_actions:
                # Penalty for redundant actions (already revealed this information)
                reward -= 0.5
                info['repeated'] = True
            else:
                # Reveal new information
                self.revealed_actions.add(action_enum)
                
                # Reward shaping: Small bonus for revealing new features
                reward += 0.2
                info['new_info'] = True
                
                # Additional reward for revealing informative features
                if action_enum == Action.MEMORY_DUMP:
                    features = CAPEFeatureExtractor.extract_memory_dump_features(self.current_report)
                    if features.get('n_anomalies', 0) > 0 or features.get('signatures_alert_count', 0) > 0:
                        reward += 1.0  # Bonus for finding suspicious indicators
                elif action_enum == Action.FOCUS_FILESYSTEM:
                    features = CAPEFeatureExtractor.extract_filesystem_features(self.current_report)
                    if features.get('n_dropped', 0) > 0:
                        reward += 0.3  # Bonus for finding dropped files
            
            # Episode length limit
            self.step_count += 1
            if self.step_count >= self.max_steps:
                self.done = True
                # Penalty for exceeding maximum steps without classification
                reward -= 5.0
                info['max_steps'] = True
                info['terminated'] = False
        
        # Update cumulative reward tracking
        self.total_reward += reward
        
        # Construct next state with updated revealed actions
        next_state = StateBuilder.build_state(
            self.current_report,
            self.revealed_actions,
            self.step_count,
            last_action=action
        )
        
        # Log state for trajectory visualization
        self.feature_history.append(next_state.copy())
        
        # Augment info dictionary with episode statistics
        info['total_reward'] = self.total_reward
        info['revealed_actions'] = list(self.revealed_actions)
        info['step_count'] = self.step_count
        
        return next_state, reward, self.done, info
    
    def get_available_actions(self) -> List[int]:
        """
        Determine legal actions in current state (action masking).
        
        Returns:
            List of valid action indices
        """
        if self.done:
            return []
            
        available = list(Action)
        
        # Constraint: Require minimum exploration before terminal actions
        if self.step_count < 3:
            available = [a for a in available if a not in TERMINAL_ACTIONS]
            
        return available
    
    def get_action_mask(self) -> List[int]:
        """
        Generate binary action mask for neural network input.
        
        Returns:
            Binary list where 1 indicates available action, 0 indicates masked
        """
        available = self.get_available_actions()
        mask = [1 if action in available else 0 for action in Action]
        return mask

# Environment Validation: Test basic functionality
if 'report_loader' in locals() and len(report_loader) > 10:
    print("\nValidating environment mechanics...")
    # Create small test subset for validation
    test_indices = list(range(min(10, len(report_loader))))
    class TestLoader:
        def __init__(self, parent_loader, indices):
            self.parent_loader = parent_loader
            self.indices = indices
            self.labels = [parent_loader.labels[i] for i in indices]
        
        def __len__(self):
            return len(self.indices)
        
        def __getitem__(self, idx):
            original_idx = self.indices[idx]
            return self.parent_loader[original_idx]
    
    test_subset_loader = TestLoader(report_loader, test_indices)
    env = CapeMalwareEnv(test_subset_loader, max_steps=5)
    state = env.reset()
    print(f"Initial state shape: {state.shape}")
    
    available_actions = env.get_available_actions()
    print(f"Available actions: {[Action(a).name for a in available_actions]}")
    
    # Execute sample trajectory
    for i in range(3):
        action = random.choice(available_actions)
        next_state, reward, done, info = env.step(action)
        print(f"Step {i}: Action={Action(action).name}, Reward={reward:.2f}, Done={done}")
        available_actions = env.get_available_actions()

# MalWhere DDDQN

State-of-the-art Double Dueling Deep Q-Network a value and advantage based reinforcement learning algorithm
This approach improves learning efficiency and mitigate overestimation bias problem, especially in domains where many actions have almost similar effect

### Implementation Details:
- **Input**: State vector (35 dimensions)
- **Hidden Layers**: 256 -> 128 neurons with ReLU activation
- **Regularization**: Dropout (0.1) to prevent overfitting
- **Action Masking**: support for illegal action elimination

In [ ]:
class DDDQN(nn.Module):
    def __init__(self, dim_states: int, dim_actions: int, dim_hidden: int = 256):
        super(DDDQN, self).__init__()
        self.features_1 = nn.Linear(dim_states, dim_hidden)
        self.features_2 = nn.Linear(dim_hidden, dim_hidden)
        self.dropout = nn.Dropout(0.1)
        self.values_1 = nn.Linear(dim_hidden, dim_hidden // 2)
        self.values_2 = nn.Linear(dim_hidden // 2, 1)
        self.advantages_1 = nn.Linear(dim_hidden, dim_hidden // 2)
        self.advantages_2 = nn.Linear(dim_hidden // 2, dim_actions)

    def forward(self, x: torch.Tensor, avilable_actions: Optional[torch.Tensor] = None) -> torch.Tensor:
        f = F.relu(self.features_1(x))
        f = F.relu(self.features_2(f))
        f = self.dropout(f)
        v = F.relu(self.values_1(f))
        V = self.values_2(v)
        a = F.relu(self.advantages_1(f))
        A = self.advantages_2(a)
        Q = V + A - torch.mean(A, dim=1, keepdim=True)
        if avilable_actions is not None:
            Q = Q.masked_fill(avilable_actions == 0, -1e9)
        return Q

# Architecture Validation: Test network construction and forward pass
print("Validating Dueling DQN architecture...")
state_dim = StateBuilder.TOTAL_FEATURE_DIM
action_dim = NUM_ACTIONS

test_network = DDDQN(state_dim, action_dim, dim_hidden=128)
print(f"Network Configuration:")
print(f"  Input dimension: {state_dim}")
print(f"  Output dimension: {action_dim}")
print(f"  Total parameters: {sum(p.numel() for p in test_network.parameters()):,}")

# Test forward pass with random input
test_state = torch.randn(1, state_dim)
test_mask = torch.ones(1, action_dim)
output = test_network(test_state, test_mask)
print(f"  Output shape: {output.shape}")
print(f"  Sample Q-values: {output[0].detach().cpu().numpy()[:5]}")

## Experience Replay: Prioritized Sampling 

Prioritized Experience Replay (PER) a technique that samples training transitions based on their temporal-difference (TD) error magnitude

### Motivation:
Standard uniform replay suffers from inefficient learning as not all transitions are equally informative. PER addresses this by:
1. **Priority Assignment**: High TD-error transitions sampled more frequently
2. **Importance Sampling**: Bias correction via importance sampling weights
3. **Adaptive Priorities**: Dynamic priority updates based on learning progress

### Implementation:
- **Priority Metric**: TD error magnitude |Q(s,a) - target|
- **Sampling Distribution**: P(i) ∝ p_i^α, where p_i is priority
- **Importance Weights**: w_i = (1/N × 1/P(i))^β for bias correction

In [ ]:
# Prioritized Experience Replay: Efficient sampling based on learning importance
class PrioritizedReplayBuffer:
    """
    Prioritized Experience Replay buffer for deep reinforcement learning.
    
    Implements the PER algorithm that samples transitions proportionally to their
    temporal-difference error, accelerating learning on informative experiences.
    
    Key Features:
    - Priority-based sampling (high TD-error transitions favored)
    - Importance sampling weights for bias correction
    - Efficient priority updates
    - Automatic capacity management with FIFO eviction
    
    References:
        Schaul et al. (2015). "Prioritized Experience Replay"
    """
    
    def __init__(self, capacity: int = 10000, alpha: float = 0.6, beta: float = 0.4):
        """
        Initialize replay buffer with prioritization parameters.
        
        Args:
            capacity: Maximum number of transitions to store
            alpha: Priority exponent (0=uniform, 1=full prioritization)
            beta: Importance sampling exponent (0=no correction, 1=full correction)
        """
        self.capacity = capacity
        self.alpha = alpha  # Controls how much prioritization is used
        self.beta = beta    # Controls importance sampling correction
        self.buffer = []
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.position = 0
        self.size = 0
        
    def push(self, state, action, reward, next_state, done, available_actions):
        """
        Store transition in buffer with maximum priority initialization.
        
        New transitions are assigned maximum priority to ensure they are sampled
        at least once before priority decay.
        
        Args:
            state: Current state vector
            action: Action taken
            reward: Reward received
            next_state: Resulting state
            done: Episode termination flag
            available_actions: Action mask for next state
        """
        max_priority = self.priorities.max() if self.size > 0 else 1.0
        
        if self.size < self.capacity:
            self.buffer.append((state, action, reward, next_state, done, available_actions))
        else:
            self.buffer[self.position] = (state, action, reward, next_state, done, available_actions)
            
        self.priorities[self.position] = max_priority
        self.position = (self.position + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)
        
    def sample(self, batch_size: int):
        """
        Sample batch of transitions with priority-based probabilities.
        
        Implements stochastic prioritization with importance sampling weights
        for bias correction during Q-learning updates.
        
        Args:
            batch_size: Number of transitions to sample
            
        Returns:
            Tuple of (states, actions, rewards, next_states, dones, masks, indices, weights)
            or None if buffer contains insufficient samples
        """
        if self.size < batch_size:
            return None
            
        # Compute sampling probabilities from priorities
        priorities = self.priorities[:self.size]
        probs = priorities ** self.alpha
        probs /= probs.sum()
        
        # Sample transition indices
        indices = np.random.choice(self.size, batch_size, p=probs, replace=False)
        
        # Retrieve sampled transitions
        samples = [self.buffer[idx] for idx in indices]
        
        # Calculate importance sampling weights for bias correction
        total = self.size
        weights = (total * probs[indices]) ** (-self.beta)
        weights /= weights.max()  # Normalize for stability
        weights = torch.FloatTensor(weights).unsqueeze(1)
        
        # Unpack transitions into batched tensors
        states, actions, rewards, next_states, dones, available_actions = zip(*samples)
        
        return (
            torch.FloatTensor(np.array(states)),
            torch.LongTensor(actions).unsqueeze(1),
            torch.FloatTensor(rewards).unsqueeze(1),
            torch.FloatTensor(np.array(next_states)),
            torch.FloatTensor(dones).unsqueeze(1),
            torch.FloatTensor(np.array(available_actions)),
            indices,
            weights
        )
        
    def update_priorities(self, indices: np.ndarray, priorities: np.ndarray):
        """
        Update priorities for sampled transitions based on TD error.
        
        Args:
            indices: Array of transition indices
            priorities: New priority values (typically |TD error|)
        """
        for idx, priority in zip(indices, priorities):
            self.priorities[idx] = priority + 1e-5  # Small epsilon prevents zero priority

# Buffer Validation: Test prioritized sampling mechanism
print("\nValidating Prioritized Replay Buffer...")
buffer = PrioritizedReplayBuffer(capacity=100)

# Populate buffer with random transitions
for i in range(10):
    buffer.push(
        state=np.random.randn(state_dim).astype(np.float32),
        action=random.randint(0, action_dim-1),
        reward=random.uniform(-1, 1),
        next_state=np.random.randn(state_dim).astype(np.float32),
        done=random.random() > 0.5,
        available_actions=np.ones(action_dim).astype(np.float32)
    )

sample = buffer.sample(5)
if sample:
    states, actions, rewards, next_states, dones, masks, indices, weights = sample
    print(f"  Sampled batch shapes: states={states.shape}, actions={actions.shape}")
    print(f"  Importance weights shape: {weights.shape}")

# Training MalWhere Agent 

- Policy (online) network: network for action selection
- Target network: sttable network for eval selected actiton
- Soft updates: tau-weighted averaging prevents abrupt target shifts
- Gradient clip: avoid exploading graidents

- Epsilon greedy: exponentially decaying exploration ratee 
- prioritized replay: efficient sampling for learning 
- action masking: legal action enforce
- loss function: smooth l1 (Huber) loss

In [ ]:
# DQN Agent: Complete reinforcement learning system with modern enhancements
class DQNAgent:
    """
    Deep Q-Network agent with Dueling architecture, prioritized replay, and soft target updates.
    
    This class encapsulates the complete training pipeline including:
    - Dual network architecture (policy + target)
    - Epsilon-greedy exploration strategy
    - Prioritized experience replay
    - Soft target network updates
    - Gradient clipping for stability
    
    Training Algorithm: Double DQN with prioritized sampling
    """
    
    def __init__(self, 
                 state_dim: int, 
                 action_dim: int,
                 lr: float = 1e-4,
                 gamma: float = 0.99,
                 tau: float = 0.005,
                 buffer_size: int = 10000):
        """
        Initialize DQN agent with neural networks and training infrastructure.
        
        Args:
            state_dim: Dimensionality of state space
            action_dim: Number of discrete actions
            lr: Learning rate for Adam optimizer
            gamma: Discount factor for future rewards
            tau: Soft update coefficient for target network
            buffer_size: Capacity of replay buffer
        """
        
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma  # Reward discount factor
        self.tau = tau      # Target network update rate
        
        # Device configuration: GPU if available, else CPU
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        
        # Neural Network Architecture: Policy and Target networks
        self.policy_net = DuelingDQN(state_dim, action_dim).to(self.device)
        self.target_net = DuelingDQN(state_dim, action_dim).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        
        # Optimizer: Adam with default parameters
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        
        # Experience Replay: Prioritized sampling buffer
        self.memory = PrioritizedReplayBuffer(buffer_size)
        
        # Exploration Strategy: Epsilon-greedy with exponential decay
        self.steps_done = 0
        self.epsilon_start = 1.0
        self.epsilon_end = 0.01
        self.epsilon_decay = 10000
        
        # Training Metrics: Track loss and exploration rate
        self.losses = []
        self.epsilons = []
        
    def select_action(self, state: np.ndarray, available_actions: List[int], training: bool = True) -> int:
        """
        Select action using epsilon-greedy policy with legal action masking.
        
        Args:
            state: Current state vector
            available_actions: List of legal action indices
            training: If True, use epsilon-greedy; if False, use greedy policy
            
        Returns:
            Selected action index
        """
        # Construct action mask for neural network
        action_mask = torch.zeros(self.action_dim, dtype=torch.float32)
        for action in available_actions:
            action_mask[action] = 1.0
        
        # Compute current epsilon value (exponential decay schedule)
        epsilon = self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
                 np.exp(-1. * self.steps_done / self.epsilon_decay)
        
        self.epsilons.append(epsilon)
        
        if training and random.random() < epsilon:
            # Exploration: Random action from available set
            return random.choice(available_actions)
        else:
            # Exploitation: Greedy action from policy network
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                mask_tensor = action_mask.unsqueeze(0).to(self.device)
                
                q_values = self.policy_net(state_tensor, mask_tensor)
                action = q_values.argmax(dim=1).item()
                
                # Safety check: Ensure selected action is legal
                if action not in available_actions:
                    action = random.choice(available_actions)
                    
            return action
    
    def update(self, batch_size: int = 128):
        """
        Perform single gradient descent step on sampled minibatch.
        
        Implements Double DQN update with prioritized experience replay and
        importance sampling weight correction.
        
        Args:
            batch_size: Number of transitions to sample for update
            
        Returns:
            Loss value if update performed, None otherwise
        """
        if self.memory.size < batch_size:
            return None
            
        # Sample prioritized minibatch from replay buffer
        batch = self.memory.sample(batch_size)
        if batch is None:
            return None
            
        states, actions, rewards, next_states, dones, next_masks, indices, weights = batch
        
        # Transfer tensors to computation device
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        next_masks = next_masks.to(self.device)
        weights = weights.to(self.device)
        
        # Compute current Q-values: Q(s, a) from policy network
        current_q = self.policy_net(states).gather(1, actions)
        
        # Compute target Q-values: r + γ * max_a' Q_target(s', a')
        with torch.no_grad():
            next_q = self.target_net(next_states, next_masks).max(1)[0].unsqueeze(1)
            target_q = rewards + (self.gamma * next_q * (1 - dones))
        
        # Compute loss with importance sampling weights (Huber loss for robustness)
        loss = F.smooth_l1_loss(current_q, target_q, reduction='none')
        loss = (loss * weights).mean()
        
        # Backpropagation and optimization
        self.optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        
        self.optimizer.step()
        
        # Update replay buffer priorities based on TD error
        with torch.no_grad():
            priorities = torch.abs(current_q - target_q).cpu().numpy() + 1e-6
            self.memory.update_priorities(indices, priorities.flatten())
        
        # Soft update of target network: θ_target ← τ*θ_policy + (1-τ)*θ_target
        self.soft_update()
        
        # Log training metrics
        self.losses.append(loss.item())
        return loss.item()
    
    def soft_update(self):
        """
        Perform soft (Polyak averaging) update of target network parameters.
        
        Gradual updates prevent instability from abrupt target shifts.
        """
        for target_param, policy_param in zip(self.target_net.parameters(), self.policy_net.parameters()):
            target_param.data.copy_(self.tau * policy_param.data + (1 - self.tau) * target_param.data)
    
    def save(self, path: str):
        """
        Persist agent state to disk for later evaluation or continued training.
        
        Args:
            path: File path for checkpoint
        """
        torch.save({
            'policy_net_state_dict': self.policy_net.state_dict(),
            'target_net_state_dict': self.target_net.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'steps_done': self.steps_done,
            'losses': self.losses,
            'epsilons': self.epsilons
        }, path)
        print(f"Model checkpoint saved to {path}")
        
    def load(self, path: str):
        """
        Load agent state from disk checkpoint.
        
        Args:
            path: File path to checkpoint
        """
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.policy_net.load_state_dict(checkpoint['policy_net_state_dict'])
        self.target_net.load_state_dict(checkpoint['target_net_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.steps_done = checkpoint['steps_done']
        self.losses = checkpoint['losses']
        self.epsilons = checkpoint['epsilons']
        print(f"Model checkpoint loaded from {path}")

# Agent Initialization and Validation
print("\nInitializing DQN Agent...")
agent = DQNAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    lr=1e-4,
    buffer_size=1000
)
print(f"Agent initialized successfully")
print(f"Policy network device: {next(agent.policy_net.parameters()).device}")

# Test action selection with random state
test_state_random = np.random.randn(state_dim).astype(np.float32)
test_available_actions = list(range(action_dim))
action = agent.select_action(test_state_random, test_available_actions, training=True)
print(f"  Test action selection: Action {action} ({Action(action).name})")

# Dataset Preparattion

### Train/Dev/Test: 80%, 10%, 10%

In [ ]:
# Dataset Preparation: Stratified splits with class balancing
def prepare_data_lazy(report_loader: LazyReportLoader, test_size=0.2, val_size=0.1):
    """
    Create balanced train/validation/test splits from lazy-loaded dataset.
    
    Implements class balancing via undersampling to address dataset imbalance,
    followed by stratified splitting to maintain class proportions.
    
    Args:
        report_loader: LazyReportLoader instance
        test_size: Proportion of data for test set
        val_size: Proportion of remaining data for validation set
        
    Returns:
        Tuple of (train_loader, val_loader, test_loader)
    """
    # Extract indices and labels for all samples
    all_indices = list(range(len(report_loader)))
    all_labels = [report_loader.labels[i] for i in all_indices]
    
    # Separate samples by class for balanced sampling
    benign_indices = [i for i, label in enumerate(all_labels) if label == 'BENIGN']
    malware_indices = [i for i, label in enumerate(all_labels) if label == 'MALWARE']
    
    print(f"Original class distribution - Benign: {len(benign_indices)}, Malware: {len(malware_indices)}")
    
    # Class Balancing: Undersample majority class (malware) to match minority class (benign)
    random.seed(SEED)
    malware_sampled = random.sample(malware_indices, len(benign_indices))
    
    # Construct balanced dataset
    balanced_indices = benign_indices + malware_sampled
    balanced_labels = ['BENIGN'] * len(benign_indices) + ['MALWARE'] * len(malware_sampled)
    
    print(f"Balanced class distribution - Benign: {len(benign_indices)}, Malware: {len(malware_sampled)}")
    
    # First split: Separate test set with stratification
    train_val_indices, test_indices = train_test_split(
        balanced_indices,
        test_size=test_size,
        stratify=balanced_labels,
        random_state=SEED
    )
    
    # Second split: Separate validation set from remaining data
    train_val_labels = [report_loader.labels[i] for i in train_val_indices]
    
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=val_size/(1-test_size),  # Adjust proportion for remaining data
        stratify=train_val_labels,
        random_state=SEED
    )
    
    # Subset Loader: Efficient wrapper for split datasets
    class SubsetLoader:
        """
        Lazy loader wrapper for dataset subsets (train/val/test).
        
        Maintains reference to parent loader and subset indices,
        enabling memory-efficient data access.
        """
        def __init__(self, parent_loader, indices):
            self.parent_loader = parent_loader
            self.indices = indices
            self.labels = [parent_loader.labels[i] for i in indices]
            
        def get_report(self, idx):
            """Get report by subset index (converts to original dataset index)"""
            original_idx = self.indices[idx]
            return self.parent_loader.get_report(original_idx)
        
        def __len__(self):
            return len(self.indices)
        
        def __getitem__(self, idx):
            original_idx = self.indices[idx]
            return self.parent_loader[original_idx]
    
    # Instantiate subset loaders for each split
    train_loader = SubsetLoader(report_loader, train_indices)
    val_loader = SubsetLoader(report_loader, val_indices)
    test_loader = SubsetLoader(report_loader, test_indices)
    
    print(f"\nDataset Split Summary:")
    print(f"  Training set: {len(train_loader)} samples")
    print(f"  Validation set: {len(val_loader)} samples")
    print(f"  Test set: {len(test_loader)} samples")
    print(f"  Training class distribution: {train_loader.labels.count('BENIGN')} benign, {train_loader.labels.count('MALWARE')} malware")
    
    return train_loader, val_loader, test_loader

# Execute dataset preparation pipeline
if 'report_loader' in locals():
    train_loader, val_loader, test_loader = prepare_data_lazy(
        report_loader, test_size=0.2, val_size=0.1
    )
    
    # Initialize environments for each dataset split
    train_env = CapeMalwareEnv(train_loader, max_steps=20)
    val_env = CapeMalwareEnv(val_loader, max_steps=20)
    test_env = CapeMalwareEnv(test_loader, max_steps=20)
    
    print("\nEnvironment Configuration:")
    print(f"  State dimension: {StateBuilder.TOTAL_FEATURE_DIM}")
    print(f"  Action space: {NUM_ACTIONS} discrete actions")
else:
    print("Error: Report loader not found. Please execute data loading cell first.")

# Training Loop and Model Optimization


- **Episodes**: 10,000 iterations
- **Update Frequency**: Gradient update every 4 steps
- **Batch Size**: 128 transitions per update
- **Validation Interval**: Every 10 episodes
- **Checkpointing**: Best model saved based on validation accuracy


In [ ]:
# Training Pipeline: Complete reinforcement learning training loop
def train_agent(agent, train_env, val_env, num_episodes=100, 
                update_freq=4, batch_size=64, save_path="/kaggle/working/dqn_cape.pth"):
    """
    Train DQN agent using experience replay and periodic validation.
    
    Implements the standard DQN training loop with enhancements:
    - Periodic validation for early stopping
    - Best model checkpointing
    - Comprehensive metric logging
    
    Args:
        agent: DQNAgent instance
        train_env: Training environment
        val_env: Validation environment
        num_episodes: Total training iterations
        update_freq: Steps between gradient updates
        batch_size: Minibatch size for updates
        save_path: Model checkpoint path
        
    Returns:
        Training history (rewards, steps, accuracies, validation accuracies)
    """
    episode_rewards = []
    episode_steps = []
    episode_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    
    for episode in range(num_episodes):
        # Episode Initialization
        state = train_env.reset()
        total_reward = 0.0
        steps = 0
        done = False
        
        # Episode Rollout: Collect trajectory
        while not done:
            available_actions = train_env.get_available_actions()
            action = agent.select_action(state, available_actions, training=True)
            
            next_state, reward, done, info = train_env.step(action)
            
            # Store transition in replay buffer
            next_available = train_env.get_available_actions() if not done else []
            next_available_mask = [1 if a in next_available else 0 for a in range(agent.action_dim)]
            
            current_mask = [1 if a in available_actions else 0 for a in range(agent.action_dim)]
            agent.memory.push(state, action, reward, next_state, done, next_available_mask)
            
            # Update state and counters
            state = next_state
            total_reward += reward
            steps += 1
            agent.steps_done += 1
            
            # Policy Improvement: Perform gradient update
            if agent.steps_done % update_freq == 0:
                loss = agent.update(batch_size)
        
        # Episode Logging
        episode_rewards.append(total_reward)
        episode_steps.append(steps)
        
        # Track classification accuracy if episode terminated with decision
        if 'correct' in info:
            episode_accuracies.append(1.0 if info['correct'] else 0.0)
        
        # Validation Phase: Periodic performance assessment
        if episode % 10 == 0 and episode > 0:
            val_accuracy = evaluate_agent_simple(agent, val_env, num_episodes=20)
            val_accuracies.append(val_accuracy)
            
            # Model Checkpointing: Save best performer
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                agent.save(save_path.replace('.pth', '_best.pth'))
        
        # Progress Reporting
        if episode % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:]) if len(episode_rewards) >= 10 else total_reward
            avg_steps = np.mean(episode_steps[-10:]) if len(episode_steps) >= 10 else steps
            
            accuracy_str = ""
            if episode_accuracies:
                avg_acc = np.mean(episode_accuracies[-10:]) if len(episode_accuracies) >= 10 else episode_accuracies[-1]
                accuracy_str = f", Train Acc: {avg_acc:.3f}"
            
            val_str = ""
            if val_accuracies:
                val_str = f", Val Acc: {val_accuracies[-1]:.3f}" if len(val_accuracies) > 0 else ""
            
            print(f"Episode {episode:4d}: Reward: {total_reward:6.2f}, Steps: {steps:2d}, "
                  f"Avg Reward: {avg_reward:6.2f}{accuracy_str}{val_str}")
    
    # Final Model Persistence
    agent.save(save_path)
    
    return episode_rewards, episode_steps, episode_accuracies, val_accuracies

def evaluate_agent_simple(agent, env, num_episodes=50):
    """
    Evaluate agent performance on validation/test environment.
    
    Runs greedy policy (no exploration) and computes classification accuracy.
    
    Args:
        agent: DQNAgent instance
        env: Evaluation environment
        num_episodes: Number of evaluation episodes
        
    Returns:
        Classification accuracy on evaluation set
    """
    correct = 0
    total = 0
    
    for _ in range(num_episodes):
        state = env.reset()
        done = False
        
        while not done:
            available_actions = env.get_available_actions()
            action = agent.select_action(state, available_actions, training=False)
            next_state, reward, done, info = env.step(action)
            state = next_state
            
            if done and 'correct' in info:
                if info['correct']:
                    correct += 1
                total += 1
    
    accuracy = correct / total if total > 0 else 0.0
    return accuracy

# Training Execution: Initialize agent and begin training
if 'train_env' in locals():
    torch.serialization.add_safe_globals([np.core.multiarray.scalar])
    print("\nStarting training...")
    agent = DQNAgent(
        state_dim=StateBuilder.TOTAL_FEATURE_DIM,
        action_dim=NUM_ACTIONS,
        lr=1e-4,
        buffer_size=100000
    )
    
    episode_rewards, episode_steps, episode_accuracies, val_accuracies = train_agent(
        agent=agent,
        train_env=train_env,
        val_env=val_env,
        num_episodes=10000,
        update_freq=4,
        batch_size=128,
        save_path="/kaggle/working/dqn_cape.pth"
    )
    
    print("\nTraining completed!")
else:
    print("Error: Training environment not initialized. Please execute data preparation cell.")

## Model Evaluation

Comprehensive evaluation metrics and visualizations 

### Evaluation Metrics:
- **Classification Performance**: Accuracy, Precision, Recall, F1-Score
- **Efficiency Metrics**: Average steps per decision, episode lengths
- **Behavioral Analysis**: Action distribution, exploration patterns

### Visualization Suite:
1. **Training Curves**: Reward convergence and accuracy trends
2. **Confusion Matrix**: Classification performance breakdown
3. **Action Distribution**: Agent decision-making patterns
4. **Episode Statistics**: Decision efficiency analysis

In [ ]:
# Evaluation Framework: Comprehensive performance analysis and visualization
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from collections import defaultdict
from typing import List, Dict, Any
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from matplotlib.ticker import MaxNLocator

# Visualization Configuration: Academic publication standards
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    try:
        plt.style.use('seaborn-whitegrid')
    except:
        plt.style.use('ggplot')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.labelsize': 12,
    'font.size': 10,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.figsize': (10, 6),
    'lines.linewidth': 2
})

def plot_training_history(rewards: List[float], accuracies: List[float], smoothing_window: int = 50):
    """
    Visualize training convergence through reward and accuracy curves.
    
    Applies moving average smoothing to highlight trends while showing raw data transparency.
    Essential for demonstrating learning stability in publications.
    
    Args:
        rewards: Episode reward history
        accuracies: Episode accuracy history
        smoothing_window: Window size for moving average
    """
    if not rewards:
        print("No training history available for visualization.")
        return

    # Apply moving average smoothing for trend visualization
    rewards_series = pd.Series(rewards)
    acc_series = pd.Series(accuracies)
    
    smooth_rewards = rewards_series.rolling(window=smoothing_window, min_periods=1).mean()
    smooth_acc = acc_series.rolling(window=smoothing_window, min_periods=1).mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Subplot 1: Episode Rewards
    ax1.plot(rewards, alpha=0.3, color='gray', label='Raw Reward')
    ax1.plot(smooth_rewards, color='#1f77b4', linewidth=2, label=f'Moving Avg (n={smoothing_window})')
    ax1.set_title('Episode Rewards over Training', fontsize=14)
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Subplot 2: Classification Accuracy
    if accuracies:
        ax2.plot(accuracies, alpha=0.3, color='gray', label='Raw Accuracy')
        ax2.plot(smooth_acc, color='#2ca02c', linewidth=2, label=f'Moving Avg (n={smoothing_window})')
        ax2.set_title('Training Classification Accuracy', fontsize=14)
        ax2.set_xlabel('Episode')
        ax2.set_ylabel('Accuracy')
        ax2.set_ylim(-0.05, 1.05)
        ax2.legend()
        ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_confusion_matrix_heatmap(y_true, y_pred, labels=['BENIGN', 'MALWARE']):
    """
    Generate publication-ready confusion matrix visualization.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
        labels: Class label names
    """
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels, cbar=False,
                annot_kws={'size': 14, 'weight': 'bold'})
    
    ax.set_title('Confusion Matrix', fontsize=14, pad=15)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()

def plot_action_distribution(actions_history: List[int], action_enum):
    """
    Visualize agent's action selection distribution.
    
    Provides insight into learned investigation strategies and decision patterns.
    
    Args:
        actions_history: Complete action history from evaluation
        action_enum: Action enumeration class
    """
    # Aggregate action frequencies
    action_counts = defaultdict(int)
    for action in actions_history:
        action_counts[action] += 1
    
    # Prepare visualization data
    action_names = [action_enum(i).name for i in range(len(action_enum))]
    counts = [action_counts[i] for i in range(len(action_enum))]
    
    df_actions = pd.DataFrame({
        'Action': action_names,
        'Frequency': counts
    })
    
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=df_actions, x='Action', y='Frequency', hue='Action', palette='viridis', ax=ax, legend=False)
    
    # Add frequency labels
    for i, v in enumerate(counts):
        ax.text(i, v + (max(counts) * 0.01), str(v), ha='center', fontweight='bold')
    
    ax.set_title('Distribution of Agent Actions During Evaluation', fontsize=14)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_ylabel('Frequency')
    ax.set_xlabel('Action Type')
    plt.tight_layout()
    plt.show()

def plot_episode_lengths(lengths: List[int]):
    """
    Visualize distribution of decision-making efficiency (steps per episode).
    
    Args:
        lengths: List of episode lengths (step counts)
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(lengths, bins=range(1, 23), kde=False, color='#d62728', edgecolor='black', alpha=0.7, ax=ax, discrete=True)
    
    ax.set_title('Distribution of Episode Lengths (Steps)', fontsize=14)
    ax.set_xlabel('Number of Steps')
    ax.set_ylabel('Count of Episodes')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

def evaluate_agent_comprehensive(agent, env, num_episodes=100):
    """
    Conduct comprehensive evaluation with full metric suite and visualizations.
    
    Implements complete evaluation pipeline including:
    - Classification metrics (accuracy, precision, recall, F1)
    - Efficiency metrics (average steps, rewards)
    - Action pattern analysis
    - Statistical visualizations
    
    Args:
        agent: Trained DQNAgent instance
        env: Evaluation environment
        num_episodes: Number of episodes for statistical reliability
        
    Returns:
        Dictionary of evaluation metrics and episode histories
    """
    results = []
    predictions = []
    true_labels = []
    all_actions = []
    episode_lengths = []
    
    print(f"Starting evaluation over {num_episodes} episodes...")
    
    for episode in range(num_episodes):
        state = env.reset()
        episode_result = EpisodeResult(
            report_id=str(episode),
            label=env.current_label,
            steps=0,
            total_reward=0.0,
            correct=False,
            actions_taken=[],
            final_state=None
        )
        
        done = False
        while not done:
            available_actions = env.get_available_actions()
            # Greedy policy for evaluation (no exploration)
            if hasattr(agent, 'select_action'):
                action = agent.select_action(state, available_actions, training=False)
            else:
                action = 0 
            
            next_state, reward, done, info = env.step(action)
            
            episode_result.actions_taken.append(action)
            episode_result.total_reward += reward
            episode_result.steps += 1
            
            state = next_state
            
            if done:
                episode_result.final_state = state
                if 'correct' in info:
                    episode_result.correct = info['correct']
                if 'prediction' in info:
                    predictions.append(info['prediction'])
                    true_labels.append(episode_result.label)
        
        # Aggregate evaluation data
        results.append(episode_result)
        all_actions.extend(episode_result.actions_taken)
        episode_lengths.append(episode_result.steps)
        
        if (episode + 1) % 20 == 0:
            print(f"  Processed {episode + 1}/{num_episodes} episodes...")
    
    # Compute Classification Metrics
    accuracy = accuracy_score(true_labels, predictions) if predictions else 0.0
    
    # Handle edge cases for binary classification metrics
    precision = precision_score(true_labels, predictions, pos_label='MALWARE', zero_division=0)
    recall = recall_score(true_labels, predictions, pos_label='MALWARE', zero_division=0)
    f1 = f1_score(true_labels, predictions, pos_label='MALWARE', zero_division=0)
    
    # Compute Efficiency Metrics
    avg_steps = np.mean([r.steps for r in results])
    avg_reward = np.mean([r.total_reward for r in results])
    
    # Display Text Report
    print("\n" + "="*50)
    print("FINAL EVALUATION REPORT")
    print("="*50)
    print(f"Total Episodes:     {num_episodes}")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"Precision (Malware):{precision:.4f}")
    print(f"Recall (Malware):   {recall:.4f}")
    print(f"F1 Score (Malware): {f1:.4f}")
    print("-" * 50)
    print(f"Average Steps:      {avg_steps:.2f}")
    print(f"Average Reward:     {avg_reward:.2f}")
    print("="*50 + "\n")

    # Generate Research Paper Visualizations
    print("Generating visualizations...")
    
    # Visualization 1: Confusion Matrix
    if true_labels and predictions:
        plot_confusion_matrix_heatmap(true_labels, predictions)
    
    # Visualization 2: Action Distribution Analysis
    if all_actions:
        plot_action_distribution(all_actions, Action)
    
    # Visualization 3: Episode Length Distribution
    if episode_lengths:
        plot_episode_lengths(episode_lengths)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'avg_steps': avg_steps,
        'avg_reward': avg_reward,
        'results': results,
        'all_actions': all_actions,
        'history': results
    }

# Evaluation Execution Pipeline
try:
    print("Initializing Evaluation...")
    
    # Verify required components
    if 'DQNAgent' in globals() and 'StateBuilder' in globals() and 'NUM_ACTIONS' in globals():
        best_agent = DQNAgent(
            state_dim=StateBuilder.TOTAL_FEATURE_DIM,
            action_dim=NUM_ACTIONS
        )
        
        # Load best trained model from checkpoint
        model_path = "/kaggle/working/dqn_cape.pth"
        if os.path.exists(model_path):
            best_agent.load(model_path)
            print("Loaded best model from disk.")
        else:
            print(f"Note: Best model not found at {model_path}.", UserWarning)
            # If no model found and an agent exists in memory from previous cells, use it
            if 'agent' in globals():
                best_agent = agent
                print("Using agent from current variable scope.")

        # Visualization 1: Training History Analysis
        if 'episode_rewards' in globals() and 'episode_accuracies' in globals():
            print("\nDisplaying Training History...")
            plot_training_history(episode_rewards, episode_accuracies)

        # Evaluation 2: Comprehensive Test Set Evaluation
        if 'test_env' in globals():
            print("\nEvaluating on Test Environment...")
            eval_metrics = evaluate_agent_comprehensive(best_agent, test_env, num_episodes=100)
        else:
            print("Error: Test environment not found. Please execute data preparation cells.")
            
    else:
        print("Error: Required classes not defined. Please execute previous cells sequentially.")

except Exception as e:
    print(f"Evaluation error encountered: {e}")
    import traceback
    traceback.print_exc()